In [1]:
import pandas as pd
movies=pd.read_csv("./tmdb_5000_movies.csv")
credits=pd.read_csv("./tmdb_5000_credits.csv")

In [2]:
print(movies.columns)
print(credits.columns)

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')
Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')


In [3]:
movies=movies.merge(credits,on='title')

In [4]:
# from collections import Counter
# print(Counter(movies))
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [5]:
#feature engineering
#genres
#id
#keywords
#overview
#cast
#crew
#movie_id


#imbalace hence not take
movies['original_language'].value_counts()

original_language
en    4510
fr      70
es      32
zh      27
de      27
hi      19
ja      16
it      14
ko      12
cn      12
ru      11
pt       9
da       7
sv       5
nl       4
fa       4
th       3
he       3
id       2
cs       2
ta       2
ro       2
ar       2
te       1
hu       1
xx       1
af       1
is       1
tr       1
vi       1
pl       1
nb       1
ky       1
no       1
sl       1
ps       1
el       1
Name: count, dtype: int64

In [6]:
movies=movies.loc[:,['movie_id','title','genres', 'keywords','overview','cast','crew']]
movies.head(1)
#overview
#cast
#crew
#movie_id]

,movie_id,title,genres,keywords,overview,cast,crew
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [7]:
#check null data
movies.isnull().sum()
#drop the null row
movies.dropna(inplace=True)
movies.isnull().sum()
#duplicated
movies.duplicated()



0       False
1       False
2       False
3       False
4       False
        ...  
4804    False
4805    False
4806    False
4807    False
4808    False
Length: 4806, dtype: bool

In [8]:
#remove id from the genres
import ast
movies['genres']
def convert(obj):
    genres=[]
    for i in ast.literal_eval(obj):
        genres.append(i['name'])
    return genres    


In [9]:
movies['genres'].iloc[1]

'[{"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 28, "name": "Action"}]'

In [10]:
movies['genres']=movies['genres'].apply(convert)
movies['keywords']=movies['keywords'].apply(convert)

In [11]:
# movies['crew'][0]
# movies['cast'][0]

In [12]:
def convert_cast(obj):
    count=0
    result=[]
    i=0
    for cast in ast.literal_eval(obj):
        if count!=3:
            result.append(cast["name"])
            count+=1
    return result    

def fetch_director(obj):
    result=[]
    for crew in ast.literal_eval(obj):
        if crew['job']== 'Director':
            result.append(crew['name'])
            break
    return result        

In [13]:
movies['cast']=movies['cast'].apply(convert_cast)
movies['crew']=movies['crew'].apply(fetch_director)


In [14]:
#remove the space between words
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])


In [15]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [16]:
movies['tags']=movies['overview']+movies['genres']+movies['cast']+movies['crew']+movies['keywords']

In [17]:
new_df = movies.drop(columns=['overview','genres','keywords','cast','crew'])

# new_df=movies.loc[:,['movie_id','title','tags']]
# new_df['tags'][0]

### Need to join the list in tag coloumn

In [18]:
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))
new_df['tags']=new_df['tags'].apply(lambda x:x.lower())

In [19]:
from nltk.stem.porter import PorterStemmer
stemmer=PorterStemmer()
stemmer.stem("noise")
def stem_util(obj):
    result=[]
    for x in obj.split():
       result.append(stemmer.stem(x))
    return " ".join(result)    


In [20]:
new_df['tags']=new_df['tags'].apply(stem_util)

In [22]:
#get tokens from the words
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter
# from nltk import word_tokenize
from nltk.corpus import stopwords
from collections import defaultdict
tokens=defaultdict(int)
import nltk
nltk.download('stopwords')
stop_set=stopwords.words('english')
for sent in new_df['tags']:
    for token in sent.split():
        if token not in stop_set:
           tokens[token]+=1

[nltk_data] Downloading package stopwords to /home/vivek/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [23]:
print(len(tokens))

43145


# Word Embedding From the text
### Stemmering
### Vectorization

In [35]:
cv = CountVectorizer(stop_words="english",max_features=5000)
vector = cv.fit_transform(new_df['tags']).toarray()
vector.shape
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vector)
new_df[new_df['title'] == 'The Lego Movie'].index[0]
def recommend(movie):
    index = new_df[new_df['title'] == movie].index[0]
    distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
    for i in distances[1:6]:
        print(new_df.iloc[i[0]].title)

In [36]:
recommend('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [37]:
print(new_df[new_df['title'] == 'The Lego Movie'].index[0])


744


In [ ]:
from gensim.models import Word2Vec
model = Word2Vec.build_vocab()